## Faiss Indexing Benchmarking

- You can run this notebook in Google Colab or any other environment that supports GPU acceleration. The notebook benchmarks the performance of Faiss GPU indexing for large-scale vector data.

In [ ]:
!pip install faiss-gpu-cu12

In [ ]:
import time
import torch
import numpy as np
import faiss

DIM = 16
N_VECTORS = 40_000_000   
CHUNK_SIZE = 5_000_000   
N_QUERIES = 100
K = 10
NLIST = 4096             
NPROBE = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Processing Device: {device}")

np.random.seed(42)
queries_np = np.random.random((N_QUERIES, DIM)).astype('float32')
queries_gpu = torch.from_numpy(queries_np).to(device)

global_bf_distances = torch.full((N_QUERIES, K), float('inf'), device=device)
global_bf_indices = torch.full((N_QUERIES, K), -1, dtype=torch.long, device=device)

res = faiss.StandardGpuResources()
quantizer = faiss.IndexFlatL2(DIM)
cpu_index_ivf = faiss.IndexIVFFlat(quantizer, DIM, NLIST, faiss.METRIC_L2)
gpu_index_ivf = faiss.index_cpu_to_gpu(res, 0, cpu_index_ivf)

print("\n--- Step 1: Training IVF Cluster Centroids on GPU ---")
start_train = time.time()
train_data = np.random.random((2_000_000, DIM)).astype('float32')
gpu_index_ivf.train(train_data)
print(f"Centroids trained in: {time.time() - start_train:.2f} seconds")

print("\n--- Step 2: Streaming Identical Chunks to Brute-Force & IVF Index ---")
start_processing = time.time()

for chunk_idx in range(0, N_VECTORS, CHUNK_SIZE):
    actual_size = min(CHUNK_SIZE, N_VECTORS - chunk_idx)
    
    chunk_np = np.random.random((actual_size, DIM)).astype('float32')
    
    gpu_index_ivf.add(chunk_np)

    chunk_gpu = torch.from_numpy(chunk_np).to(device)
    q_norms = (queries_gpu ** 2).sum(dim=1, keepdim=True)
    c_norms = (chunk_gpu ** 2).sum(dim=1, keepdim=True).T
    distances = q_norms + c_norms - 2.0 * torch.mm(queries_gpu, chunk_gpu.T)
    
    local_dist, local_idx = torch.topk(distances, K, dim=1, largest=False)
    local_idx += chunk_idx
    
    combined_dist = torch.cat([global_bf_distances, local_dist], dim=1)
    combined_idx = torch.cat([global_bf_indices, local_idx], dim=1)
    
    global_bf_distances, sorted_meta_idx = torch.topk(combined_dist, K, dim=1, largest=False)
    global_bf_indices = torch.gather(combined_idx, 1, sorted_meta_idx)
    
    del chunk_gpu, distances
    torch.cuda.empty_cache()
    print(f"  Processed chunk: {chunk_idx + actual_size:,} / {N_VECTORS:,}")

print(f"Data loading and Brute Force calculations finished in: {time.time() - start_processing:.2f} seconds")

print("\n--- Step 3: Running IVF Search Optimization Pass ---")
gpu_index_ivf.nprobe = NPROBE
start_search = time.time()
ivf_distances, ivf_indices = gpu_index_ivf.search(queries_np, K)
ivf_search_time = time.time() - start_search

# Calculate precise overlap alignment
bf_indices_cpu = global_bf_indices.cpu().numpy()
correct_matches = 0
for i in range(N_QUERIES):
    true_set = set(bf_indices_cpu[i])
    ivf_set = set(ivf_indices[i])
    correct_matches += len(true_set.intersection(ivf_set))
ivf_recall = correct_matches / (N_QUERIES * K)

print("\n================== CORRECTED EXPERIMENT MATRIX ==================")
print(f"IVF-Flat (GPU) Search Time : {ivf_search_time:.4f} seconds")
print(f"Aligned IVF Recall Accuracy : {ivf_recall * 100:.2f}%")
print("=================================================================")


Active Processing Device: cuda

--- Step 1: Training IVF Cluster Centroids on GPU ---
Centroids trained in: 4.33 seconds

--- Step 2: Streaming Identical Chunks to Brute-Force & IVF Index ---
  Processed chunk: 5,000,000 / 40,000,000
  Processed chunk: 10,000,000 / 40,000,000
  Processed chunk: 15,000,000 / 40,000,000
  Processed chunk: 20,000,000 / 40,000,000
  Processed chunk: 25,000,000 / 40,000,000
  Processed chunk: 30,000,000 / 40,000,000
  Processed chunk: 35,000,000 / 40,000,000
  Processed chunk: 40,000,000 / 40,000,000
Data loading and Brute Force calculations finished in: 46.16 seconds

--- Step 3: Running IVF Search Optimization Pass ---

================== CORRECTED EXPERIMENT MATRIX ==================
IVF-Flat (GPU) Search Time : 0.0908 seconds
Aligned IVF Recall Accuracy : 100.00%


In [ ]:

torch.cuda.synchronize()
start_bf_search = time.time()

q_norms = (queries_gpu ** 2).sum(dim=1, keepdim=True)
sample_chunk = torch.rand((CHUNK_SIZE, DIM), dtype=torch.float32, device=device)
_ = q_norms + (sample_chunk ** 2).sum(dim=1, keepdim=True).T - 2.0 * torch.mm(queries_gpu, sample_chunk.T)

torch.cuda.synchronize()
pure_bf_search_time = (time.time() - start_bf_search) * (N_VECTORS / CHUNK_SIZE)

bf_indices_cpu = global_bf_indices.cpu().numpy()

nprobe_test_values = [1, 4, 16, 32, 64, 128, 256]

print("\n=========================================================================")
print(f"       VEC-SEARCH BENCHMARK MATRIX (40M Corpus | Dim: {DIM} | Top-K: {K})")
print("=========================================================================")
print(f" Strategy          | NPROBE  | Search Time (s) | Recall  | Speedup vs BF")
print("-------------------------------------------------------------------------")
print(f" Brute Force (GPU) | N/A     | {pure_bf_search_time:.6f}      | 100.00% | 1.00x (Baseline)")
print("-------------------------------------------------------------------------")

for np_val in nprobe_test_values:
    # Setting the target search depth dynamically
    gpu_index_ivf.nprobe = np_val
    
    # Tracking pure search latency
    start_ivf_search = time.time()
    ivf_distances, ivf_indices = gpu_index_ivf.search(queries_np, K)
    ivf_search_time = time.time() - start_ivf_search
    
    correct_matches = 0
    for i in range(N_QUERIES):
        true_set = set(bf_indices_cpu[i])
        ivf_set = set(ivf_indices[i])
        correct_matches += len(true_set.intersection(ivf_set))
    ivf_recall = (correct_matches / (N_QUERIES * K)) * 100
    
    speedup = pure_bf_search_time / ivf_search_time if ivf_search_time > 0 else 0
    
    print(f" IVF-Flat (GPU)    | {np_val:<7} | {ivf_search_time:.6f}      | {ivf_recall:>6.2f}% | {speedup:.2f}x")

print("=========================================================================")



       VEC-SEARCH BENCHMARK MATRIX (40M Corpus | Dim: 16 | Top-K: 10)
 Strategy          | NPROBE  | Search Time (s) | Recall  | Speedup vs BF
-------------------------------------------------------------------------
 Brute Force (GPU) | N/A     | 0.755268      | 100.00% | 1.00x (Baseline)
-------------------------------------------------------------------------
 IVF-Flat (GPU)    | 1       | 0.001164      |  31.90% | 649.01x
 IVF-Flat (GPU)    | 4       | 0.003546      |  68.40% | 212.99x
 IVF-Flat (GPU)    | 16      | 0.006732      |  92.40% | 112.19x
 IVF-Flat (GPU)    | 32      | 0.011387      |  98.60% | 66.33x
 IVF-Flat (GPU)    | 64      | 0.026091      |  99.90% | 28.95x
 IVF-Flat (GPU)    | 128     | 0.043739      | 100.00% | 17.27x
 IVF-Flat (GPU)    | 256     | 0.087729      | 100.00% | 8.61x
